In [ ]:
"""
Bài tập Khoa học Dữ liệu - Phân cụm sinh viên K58KTP
Pipeline: Đọc dữ liệu → Làm sạch → Phân cụm K-Means → Trực quan hóa (2 biểu đồ) → Xuất kết quả
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CẤU HÌNH ĐƯỜNG DẪN HỆ THỐNG (ĐÃ SỬA LỖI)
# ─────────────────────────────────────────────
# Bạn hãy điều chỉnh đường dẫn file đầu vào phù hợp với máy của bạn
INPUT_FILE = r'D:\BAITAP\KHDL\BT_Phan_Cum\TỔNG HỢP ĐIỂM K58KTP.xlsx' 

# Đường dẫn lưu trữ đầu ra (Tạo thư mục outputs nếu chưa có)
OUTPUT_DIR = './outputs'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

CHART_PATH = os.path.join(OUTPUT_DIR, 'bieu_do_phan_cum.png')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'Phan_Cum_SinhVien_K58KTP.xlsx')

# ─────────────────────────────────────────────
# BƯỚC 1: ĐỌC VÀ LÀM SẠCH DỮ LIỆU
# ─────────────────────────────────────────────
print("=" * 60)
print("BƯỚC 1: ĐỌC VÀ LÀM SẠCH DỮ LIỆU")
print("=" * 60)

# Đọc file CSV đầu vào (sử dụng csv thay vì read_excel do định dạng file của bạn)
df_raw = pd.read_excel(INPUT_FILE, header=None)

# --- Trích xuất thông tin sinh viên ---
# Dựa trên cấu trúc file: Hàng 1 (index 1) là MSSV, Hàng 2 (index 2) là Tên sinh viên
mssv_list = df_raw.iloc[1, 3:].tolist()       
name_list = df_raw.iloc[2, 3:].tolist()       
mon_list  = df_raw.iloc[4:, 1].tolist()       # Từ hàng index 3 trở đi là các môn học
ten_mon   = df_raw.iloc[4:, 2].tolist()       

print(f"  Tổng sinh viên ban đầu : {len(mssv_list)}")
print(f"  Tổng môn học           : {len(mon_list)}")

# --- Lấy ma trận điểm ---
score_matrix = df_raw.iloc[4:, 3:].copy()
score_matrix.columns = range(len(mssv_list))
score_matrix.index   = range(len(mon_list))

# Chuyển tất cả về số, ép các giá trị lỗi thành NaN
score_numeric = score_matrix.apply(pd.to_numeric, errors='coerce')

# Loại điểm không hợp lệ (nằm ngoài thang điểm hệ 4 từ 0.0 đến 4.0)
score_clean = score_numeric.where((score_numeric >= 0) & (score_numeric <= 4.0))

print(f"\n  Trước làm sạch: {score_numeric.stack().shape[0]} đầu điểm")
print(f"  Sau làm sạch  : {score_clean.stack().shape[0]} đầu điểm hợp lệ")
print(f"  Khoảng điểm   : {score_clean.stack().min():.2f} - {score_clean.stack().max():.2f}")


# ─────────────────────────────────────────────
# BƯỚC 2: XÂY DỰNG DATAFRAME SINH VIÊN
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("BƯỚC 2: TÍNH TOÁN ĐẶC TRƯNG SINH VIÊN")
print("=" * 60)

records = []
for i, (mssv, name) in enumerate(zip(mssv_list, name_list)):
    col = score_clean.iloc[:, i].dropna()
    if len(col) == 0:
        continue  # Bỏ qua sinh viên không có điểm nào hợp lệ

    records.append({
        'MSSV'           : str(mssv),
        'Ho_Ten'         : str(name),
        'Diem_TB'        : round(col.mean(), 4),
        'Diem_Max'       : col.max(),
        'Diem_Min'       : col.min(),
        'Do_Lech_Chuan'  : round(col.std(), 4),
        'So_Mon_Co_Diem' : len(col),
        'Ti_Le_Mon_Gioi' : round((col >= 3.5).sum() / len(col), 4),   # Môn hệ 4 >= 3.5 là Giỏi/Xuất sắc
        'Ti_Le_Mon_Yeu'  : round((col < 2.0).sum() / len(col), 4),    # Môn hệ 4 < 2.0 là Yếu
    })

df = pd.DataFrame(records)
print(f"  Số sinh viên hợp lệ   : {len(df)}")
print(f"  Điểm TB trung bình lớp: {df['Diem_TB'].mean():.3f}")
print(f"  Điểm TB cao nhất      : {df['Diem_TB'].max():.3f}")
print(f"  Điểm TB thấp nhất     : {df['Diem_TB'].min():.3f}")


# ─────────────────────────────────────────────
# BƯỚC 3: PHÂN CỤM K-MEANS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("BƯỚC 3: PHÂN CỤM K-MEANS (k=3)")
print("=" * 60)

FEATURES = ['Diem_TB', 'Diem_Max', 'Diem_Min', 'Do_Lech_Chuan', 'Ti_Le_Mon_Gioi', 'Ti_Le_Mon_Yeu']

X = df[FEATURES].fillna(0).values

# Chuẩn hóa dữ liệu (StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Chạy mô hình K-Means với k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
raw_labels = kmeans.fit_predict(X_scaled)

# Định vị và gán nhãn cụm theo điểm trung bình từ thấp -> cao để ánh xạ đúng nhóm học lực
cluster_means = {}
for c in range(3):
    cluster_means[c] = df.loc[raw_labels == c, 'Diem_TB'].mean()

sorted_clusters = sorted(cluster_means, key=cluster_means.get)
label_map = {
    sorted_clusters[0]: ('Trung bình / Yếu', '#E74C3C'),   # Màu đỏ
    sorted_clusters[1]: ('Khá / Giỏi',       '#F39C12'),   # Màu cam
    sorted_clusters[2]: ('Xuất sắc',          '#27AE60'),   # Màu xanh lá
}

df['Cum_Goc']   = raw_labels
df['Cum_Ten']   = [label_map[l][0] for l in raw_labels]
df['Cum_Mau']   = [label_map[l][1] for l in raw_labels]
df['Cum_Ma']    = df['Cum_Ten'].map({'Trung bình / Yếu': 1, 'Khá / Giỏi': 2, 'Xuất sắc': 3})

# Thống kê kết quả phân cụm ra màn hình
print("\n  Kết quả phân cụm:")
for ten, grp in df.groupby('Cum_Ten', sort=False):
    print(f"  • {ten:20s}: {len(grp):3d} SV | Điểm TB Cụm = {grp['Diem_TB'].mean():.3f}")

sil_final = silhouette_score(X_scaled, raw_labels)
print(f"\n  Silhouette Score (k=3) = {sil_final:.4f}")


# ─────────────────────────────────────────────
# BƯỚC 4: TRỰC QUAN HÓA (CHỈ GIỮ LẠI 2 BIỂU ĐỒ)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("BƯỚC 4: VẼ BIỂU ĐỒ TRỰC QUAN")
print("=" * 60)

COLOR_MAP = {'Trung bình / Yếu': '#E74C3C', 'Khá / Giỏi': '#F39C12', 'Xuất sắc': '#27AE60'}
ORDER = ['Trung bình / Yếu', 'Khá / Giỏi', 'Xuất sắc']

# Thiết lập layout chứa đúng 2 biểu đồ (1 hàng, 2 cột)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#F8F9FA')
fig.suptitle('Phân Cụm Thống Kê Kết Quả Học Tập Sinh Viên K58KTP', fontsize=15, fontweight='bold', y=0.98)

# ── Biểu đồ 1: Số lượng SV mỗi cụm (Bar Chart) ──
ax1 = axes[0]
counts = [df[df['Cum_Ten'] == t].shape[0] for t in ORDER]
bars = ax1.bar(ORDER, counts,
               color=[COLOR_MAP[t] for t in ORDER],
               edgecolor='white', linewidth=1.5, width=0.45)

for bar, cnt in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{cnt} SV\n({cnt/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontweight='bold', fontsize=10)

ax1.set_title('Số lượng Sinh viên thuộc từng nhóm', fontweight='bold', pad=10)
ax1.set_ylabel('Số lượng sinh viên')
ax1.set_ylim(0, max(counts) * 1.2)
ax1.set_xticks(range(len(ORDER)))
ax1.set_xticklabels(ORDER, fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_facecolor('white')

# ── Biểu đồ 2: Phân cụm trên không gian PCA 2D (Scatter Plot) ──
ax2 = axes[1]
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

for ten in ORDER:
    sub = df[df['Cum_Ten'] == ten]
    ax2.scatter(sub['PCA1'], sub['PCA2'], c=COLOR_MAP[ten],
                label=ten, alpha=0.85, s=90, edgecolors='white', linewidths=0.6)

ax2.set_title('Trực quan hóa Phân cụm bằng thuật toán PCA 2D', fontweight='bold', pad=10)
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax2.legend(fontsize=9, loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('white')

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(CHART_PATH, dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
plt.close()
print(f"  ✓ Đã lưu thành công biểu đồ: {CHART_PATH}")


# ─────────────────────────────────────────────
# BƯỚC 5: XUẤT KẾT QUẢ RA EXCEL (OPENPYXL)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("BƯỚC 5: XUẤT KẾT QUẢ RA FILE EXCEL")
print("=" * 60)

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

wb = Workbook()

# --- Sheet 1: Chi tiết danh sách phân cụm ---
ws1 = wb.active
ws1.title = "Danh Sách Phân Cụm"

HEADER_FILL = PatternFill("solid", fgColor="1A3A5C")
HEADER_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=11)
thin = Side(style='thin', color='CCCCCC')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

headers = ['STT', 'MSSV', 'Họ và Tên', 'Điểm TB',
           'Điểm Cao Nhất', 'Điểm Thấp Nhất', 'Độ Lệch Chuẩn',
           'Số Môn', '% Môn Giỏi (≥3.5)', '% Môn Yếu (<2.0)',
           'Cụm Phân Loại']

col_widths = [6, 18, 25, 10, 14, 14, 14, 8, 18, 18, 22]

ws1.row_dimensions[1].height = 30
for j, (h, w) in enumerate(zip(headers, col_widths), 1):
    cell = ws1.cell(row=1, column=j, value=h)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = border
    ws1.column_dimensions[get_column_letter(j)].width = w

CUM_FILLS = {
    'Trung bình / Yếu': PatternFill("solid", fgColor="FDECEA"),
    'Khá / Giỏi':       PatternFill("solid", fgColor="FEF9E7"),
    'Xuất sắc':          PatternFill("solid", fgColor="E9F7EF"),
}
CUM_FONTS = {
    'Trung bình / Yếu': Font(color="C0392B", bold=True, name="Arial", size=10),
    'Khá / Giỏi':       Font(color="D68910", bold=True, name="Arial", size=10),
    'Xuất sắc':          Font(color="1E8449", bold=True, name="Arial", size=10),
}

df_export = df.sort_values(['Cum_Ma', 'Diem_TB'], ascending=[True, False]).reset_index(drop=True)

for idx, row in df_export.iterrows():
    r = idx + 2
    cum = row['Cum_Ten']
    row_fill = CUM_FILLS[cum]

    vals = [
        idx + 1,
        row['MSSV'],
        row['Ho_Ten'],
        round(row['Diem_TB'], 3),
        row['Diem_Max'],
        row['Diem_Min'],
        round(row['Do_Lech_Chuan'], 3),
        int(row['So_Mon_Co_Diem']),
        f"{row['Ti_Le_Mon_Gioi']*100:.1f}%",
        f"{row['Ti_Le_Mon_Yeu']*100:.1f}%",
        cum,
    ]
    for j, v in enumerate(vals, 1):
        cell = ws1.cell(row=r, column=j, value=v)
        cell.fill = row_fill
        cell.border = border
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.font = Font(name="Arial", size=10)
    ws1.cell(row=r, column=11).font = CUM_FONTS[cum]

ws1.freeze_panes = 'A2'
ws1.auto_filter.ref = f"A1:K{len(df_export)+1}"

# --- Sheet 2: Thống kê tổng hợp & Đính kèm biểu đồ ---
ws2 = wb.create_sheet("Thống Kê Tổng Hợp")

ws2.merge_cells('A1:E1')
title_cell = ws2['A1']
title_cell.value = "THỐNG KÊ PHÂN CỤM SINH VIÊN K58KTP"
title_cell.font = Font(bold=True, size=14, color="1A3A5C", name="Arial")
title_cell.alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 30

stats_headers = ['Cụm', 'Số Sinh Viên', 'Tỷ Lệ (%)', 'Điểm TB Cụm', 'Điểm TB Cao Nhất', 'Điểm TB Thấp Nhất']
for j, h in enumerate(stats_headers, 1):
    cell = ws2.cell(row=3, column=j, value=h)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = border
    ws2.column_dimensions[get_column_letter(j)].width = [22, 14, 12, 14, 18, 18][j-1]

for i, ten in enumerate(ORDER):
    sub = df[df['Cum_Ten'] == ten]
    r = i + 4
    row_data = [
        ten,
        len(sub),
        f"{len(sub)/len(df)*100:.1f}%",
        f"{sub['Diem_TB'].mean():.3f}",
        f"{sub['Diem_TB'].max():.3f}",
        f"{sub['Diem_TB'].min():.3f}",
    ]
    fill = CUM_FILLS[ten]
    for j, v in enumerate(row_data, 1):
        cell = ws2.cell(row=r, column=j, value=v)
        cell.fill = fill
        cell.border = border
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.font = Font(name="Arial", size=11)
    ws2.cell(row=r, column=1).font = CUM_FONTS[ten]

# Hàng tổng cộng
r_total = len(ORDER) + 4
ws2.cell(row=r_total, column=1, value="TỔNG CỘNG").font = Font(bold=True, name="Arial", size=11)
ws2.cell(row=r_total, column=2, value=len(df)).font = Font(bold=True, name="Arial", size=11)
ws2.cell(row=r_total, column=3, value="100%").font = Font(bold=True, name="Arial", size=11)
ws2.cell(row=r_total, column=4, value=f"{df['Diem_TB'].mean():.3f}").font = Font(bold=True, name="Arial", size=11)
for j in range(1, 7):
    ws2.cell(row=r_total, column=j).fill = PatternFill("solid", fgColor="D5DBDB")
    ws2.cell(row=r_total, column=j).border = border
    ws2.cell(row=r_total, column=j).alignment = Alignment(horizontal='center')

# Ghi chú phương pháp luận học máy
ws2['A8'] = "Phương pháp phân cụm:"
ws2['A8'].font = Font(bold=True, name="Arial", size=10)
notes = [
    ("A9",  f"• Thuật toán: K-Means Clustering (k=3)"),
    ("A10", f"• Danh sách đặc trưng: {', '.join(FEATURES)}"),
    ("A11", f"• Chuẩn hóa: StandardScaler (z-score)"),
    ("A12", f"• Chỉ số Silhouette Score đánh giá mô hình: {sil_final:.4f}"),
]
for cell_ref, text in notes:
    ws2[cell_ref] = text
    ws2[cell_ref].font = Font(name="Arial", size=10)
ws2.merge_cells('A8:E8')
for ref, _ in notes:
    ws2.merge_cells(f'{ref}:E{ref[1:]}')

# Nhúng biểu đồ trực quan đã làm gọn vào Excel dưới dạng ảnh
if os.path.exists(CHART_PATH):
    img = XLImage(CHART_PATH)
    img.width  = 780   # Điều chỉnh lại kích thước ảnh cho vừa vặn layout excel mới
    img.height = 340
    ws2.add_image(img, 'A14')

wb.save(OUTPUT_PATH)
print(f"  ✓ Đã lưu file báo cáo kết quả Excel: {OUTPUT_PATH}")
print("\n[THÀNH CÔNG] Pipeline xử lý và phân cụm hoàn tất!")

BƯỚC 1: ĐỌC VÀ LÀM SẠCH DỮ LIỆU
  Tổng sinh viên ban đầu : 71
  Tổng môn học           : 52


ValueError: Length mismatch: Expected axis has 70 elements, new values have 71 elements